# Municipality split transformer (prototype)

This notebook calculates the ratio of energy used in each municipality of Sweden (290). It is a rough prototype without much sophisticated thought.

The data used is energy use data from SCB, downloaded via their API. All energy sources are used.

We average over 10 years to catch at least a few reported numbers from all municipalities.

### Possible improvements

- Batch the API call so that we can download all data (users and energy sources)
- Do not include renewable energy sources in the calculation as these are less likely to be replaced by electricity
- Look at the trends in energy use for each municipality
- Find the responsible person at SCB and investiag

In [1]:
# Imports and general variables

import pandas as pd 
import numpy as np
import requests
from io import StringIO

In [7]:
# Load the data from SCB API

url = "https://api.scb.se/OV0104/v1/doris/sv/ssd/START/EN/EN0203/EN0203A/SlutAnvSektor"
query = {
  "query": [
    {
      "code": "Region",
      "selection": {
        "filter": "item",
        "values": [
            "0114","0115","0117","0120","0123","0125","0126","0127","0128","0136","0138","0139","0140","0160","0162","0163","0180","0181","0182","0183","0184",
            "0186","0187","0188","0191","0192","0305","0319","0330","0331","0360","0380","0381","0382","0428","0461","0480","0481","0482","0483","0484","0486",
            "0488","0509","0512","0513","0560","0561","0562","0563","0580","0581","0582","0583","0584","0586","0604","0617","0642","0643","0662","0665","0680",
            "0682","0683","0684","0685","0686","0687","0760","0761","0763","0764","0765","0767","0780","0781","0821","0834","0840","0860","0861","0862","0880",
            "0881","0882","0883","0884","0885","0980","1060","1080","1081","1082","1083","1214","1230","1231","1233","1256","1257","1260","1261","1262","1263",
            "1264","1265","1266","1267","1270","1272","1273","1275","1276","1277","1278","1280","1281","1282","1283","1284","1285","1286","1287","1290","1291",
            "1292","1293","1315","1380","1381","1382","1383","1384","1401","1402","1407","1415","1419","1421","1427","1430","1435","1438","1439","1440","1441",
            "1442","1443","1444","1445","1446","1447","1452","1460","1461","1462","1463","1465","1466","1470","1471","1472","1473","1480","1481","1482","1484",
            "1485","1486","1487","1488","1489","1490","1491","1492","1493","1494","1495","1496","1497","1498","1499","1715","1730","1737","1760","1761","1762",
            "1763","1764","1765","1766","1780","1781","1782","1783","1784","1785","1814","1860","1861","1862","1863","1864","1880","1881","1882","1883","1884",
            "1885","1904","1907","1960","1961","1962","1980","1981","1982","1983","1984","2021","2023","2026","2029","2031","2034","2039","2061","2062","2080",
            "2081","2082","2083","2084","2085","2101","2104","2121","2132","2161","2180","2181","2182","2183","2184","2260","2262","2280","2281","2282","2283",
            "2284","2303","2305","2309","2313","2321","2326","2361","2380","2401","2403","2404","2409","2417","2418","2421","2422","2425","2460","2462","2463",
            "2480","2481","2482","2505","2506","2510","2513","2514","2518","2521","2523","2560","2580","2581","2582","2583","2584"
        ]
      }
    },
    {
      "code": "Forbrukningskategri",
      "selection": {
        "filter": "item",
        "values": ["999"
        ]
      }
    },
    {
      "code": "Bransle",
      "selection": {
        "filter": "item",
        "values": ["905","910","915","920","925","930","14","16","955"]
      }
    }
  ],
  "response": {
    "format": "csv"
  }
}

##  Make the request (POST)
response = requests.post(url, json=query)
if response.status_code == 200:
    csv_data = StringIO(response.text)
    response_csv = pd.read_csv(csv_data)    
else:
    print(f"Error: {response.status_code}")

## Format dataframe
energy_data = response_csv.copy()
energy_data['Kommun'] = energy_data['region'].apply(lambda x: x.split(' ', 1)[0]) # Split the municipality code and the name
energy_data.rename(columns={'Kommun': 'municipality'}, inplace=True)
energy_data.set_index('municipality', inplace=True) # Set the code to index
energy_data = energy_data.drop(columns=['region'])
energy_data.loc[:, ~energy_data.columns.isin(['förbrukarkategori', 'bränsletyp'])] = ( # Make all numbers float
    energy_data.loc[:, ~energy_data.columns.isin(['förbrukarkategori', 'bränsletyp'])]
    .replace("..", np.nan)
    .astype(float)
)
energy_data.rename( # Rename the columns
    columns=lambda col: col.split()[-1] if col.startswith('Slutanvändning (MWh)') else col,
    inplace=True
)

In [8]:
energy_data

,förbrukarkategori,bränsletyp,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022
municipality,,,,,,,,,,,,,,,,
0114,totalt,flytande (icke förnybara),NaN,NaN,NaN,NaN,NaN,NaN,NaN,383525.0,417524.0,NaN,NaN,359363.0,403955.0,373113.0
0114,totalt,fast (icke förnybara),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0114,totalt,gas (icke förnybara),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0
0114,totalt,flytande (förnybara),0.0,NaN,NaN,28202.0,35663.0,31250.0,31900.0,NaN,63947.0,NaN,NaN,NaN,NaN,NaN
0114,totalt,fast (förnybara),5148.0,7413.0,7183.0,6917.0,6644.0,6161.0,6212.0,6257.0,6247.0,5311.0,5266.0,5033.0,5464.0,4951.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2584,totalt,fast (förnybara),30535.0,19220.0,18625.0,17934.0,NaN,NaN,16107.0,16222.0,16196.0,13771.0,13654.0,13048.0,14168.0,NaN
2584,totalt,gas (förnybara),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2584,totalt,fjärrvärme,201934.0,222717.0,193862.0,219527.0,199077.0,198257.0,NaN,187140.0,205236.0,200236.0,210367.0,201870.0,225778.0,218166.0


In [9]:
total_energy = energy_data[energy_data['bränsletyp'] == 'totalt'].drop(columns=['förbrukarkategori', 'bränsletyp'])
total_energy['last_three_avg'] = total_energy.loc[:,['2020', '2021', '2022']].mean(axis=1)
total_energy['last_three_avg_ratio'] = total_energy['last_three_avg']/total_energy['last_three_avg'].sum()

total_energy['last_five_avg'] = total_energy.loc[:,['2018', '2019', '2020', '2021', '2022']].mean(axis=1)
total_energy['last_five_avg_ratio'] = total_energy['last_five_avg']/total_energy['last_five_avg'].sum()

total_energy['last_ten_avg'] = total_energy.loc[:,['2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022']].mean(axis=1)
total_energy['last_ten_avg_ratio'] = total_energy['last_ten_avg']/total_energy['last_ten_avg'].sum()


total_energy

,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,last_three_avg,last_three_avg_ratio,last_five_avg,last_five_avg_ratio,last_ten_avg,last_ten_avg_ratio
municipality,,,,,,,,,,,,,,,,,,,,
0114,851649.0,1141725.0,1146621.0,1057923.0,989968.0,947056.0,971113.0,981275.0,1022457.0,1109035.0,1035771.0,919248.0,1036384.0,969920.0,975184.0,0.002603,1014071.6,0.002645,998222.7,0.002583
0115,519215.0,NaN,478998.0,485525.0,466077.0,455545.0,479062.0,NaN,503530.0,368043.0,501336.0,455744.0,592097.0,544843.0,530894.666667,0.001417,492412.6,0.001284,485141.888889,0.001255
0117,NaN,727317.0,NaN,687356.0,677491.0,674859.0,672824.0,699661.0,704905.0,711171.0,685362.0,654184.0,764044.0,772108.0,730112.0,0.001949,717373.8,0.001871,701660.9,0.001815
0120,815094.0,NaN,814429.0,828626.0,749005.0,721654.0,NaN,NaN,849582.0,NaN,NaN,766097.0,874826.0,766771.0,802564.666667,0.002143,802564.666667,0.002093,787989.166667,0.002039
0123,1410171.0,1533428.0,1419030.0,1472495.0,1445939.0,1487157.0,1496411.0,NaN,1563300.0,NaN,1409807.0,1344046.0,1454781.0,1363034.0,1387287.0,0.003703,1392917.0,0.003633,1445559.375,0.00374
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2580,11290315.0,NaN,NaN,NaN,NaN,12974969.0,8392736.0,NaN,13162220.0,12330806.0,12543672.0,12569140.0,12947999.0,12729379.0,12748839.333333,0.034034,12624199.2,0.032926,12206365.125,0.03158
2581,5544203.0,5292548.0,5361576.0,5825464.0,5362481.0,5882135.0,5901120.0,5881560.0,5822948.0,6024983.0,6383130.0,NaN,6322047.0,6128010.0,6225028.5,0.016618,6214542.5,0.016208,5967601.555556,0.015439
2582,NaN,996443.0,892808.0,885980.0,940127.0,996802.0,1055037.0,1053358.0,972260.0,1130837.0,1057331.0,1042927.0,1259698.0,1710100.0,1337575.0,0.003571,1240178.6,0.003235,1121847.7,0.002902


In [10]:
municipality_energy_split = total_energy['last_ten_avg_ratio']
municipality_energy_split.rename('ratio', inplace=True)
municipality_energy_split.to_csv('municipality_energy_split.csv')
